# ETL-08 · Perfil 2022

- **Etapa:** ETL-08, Perfil de 2022
- **Responsável:** Guilherme
- **Issue:** #ETL-08
- **Entrada:** `data/pessoas_*-2022.csv`
- **Saída:** `docs/perfil/perfil_2022.md`

## 1. Setup

In [1]:
from pathlib import Path
import pandas as pd

# Raiz do repositório: sobe de pasta em pasta até achar a pasta .git
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".git").exists())

DADOS  = RAIZ / "data"                      # CSVs brutos (estão no Git)

ENTRADA = DADOS      # pasta com os CSVs brutos
SAIDA   = RAIZ / "docs" / "perfil" / "perfil_2022.md"

In [2]:
import sys

# adiciona a pasta src/ à lista de lugares, para o Python achar o perfil.py
sys.path.append(str(RAIZ / "src"))

# traz de src/perfil.py: perfil() calcula as 10 perguntas
# e perfil_para_markdown() grava o resultado num arquivo .md
from perfil import perfil, perfil_para_markdown

## 2. Carregar a entrada

In [3]:
arquivos = [DADOS / f"pessoas_{mes:02d}-2022.csv" for mes in range(1, 13)]

lista = []
for a in arquivos:
    d = pd.read_csv(a, encoding='latin-1', sep=';')
    d['arquivo_origem'] = a.name   # nome do arquivo de origem
    lista.append(d)

df = pd.concat(lista, ignore_index=True)   # junta os 12 meses
print('shape:', df.shape)
df.head()

shape: (266128, 32)


,id_sinistro,id_pessoa,id_veiculo,cod_ibge,municipio,regiao_administrativa,tipo_via,tipo_veiculo_vitima,tipo_de_vitima,modo_transporte_vitima,sexo,idade,gravidade_lesao,faixa_etaria_demografica,faixa_etaria_legal,profissao,grau_de_instrucao,nacionalidade,data_sinistro,ano_sinistro,mes_sinistro,dia_sinistro,ano_mes_sinistro,data_obito,ano_obito,mes_obito,dia_obito,ano_mes_obito,local_obito,local_via,tempo_sinistro_obito,arquivo_origem
0,1566901,459976,596884.0,3507001,BOITUVA,SOROCABA,ESTRADAS E RODOVIAS,MOTOCICLETA,CONDUTOR,MOTOCICLETA,MASCULINO,31.0,LEVE,30 a 34,30-34,AUXILIAR GERAL,NaN,BRASILEIRA,01/01/2022,2022,1,1,2022/01,NaN,NaN,NaN,NaN,NaN,NaN,PUBLICO,NaN,pessoas_01-2022.csv
1,1837366,747885,785950.0,3541000,PRAIA GRANDE,BAIXADA SANTISTA,VIAS URBANAS,MOTOCICLETA,PASSAGEIRO,MOTOCICLETA,MASCULINO,17.0,LEVE,15 a 19,0-17,MOTO-BOY,NaN,BRASILEIRA,01/01/2022,2022,1,1,2022/01,NaN,NaN,NaN,NaN,NaN,NaN,PUBLICO,NaN,pessoas_01-2022.csv
2,1357207,216719,217385.0,3551009,SAO VICENTE,BAIXADA SANTISTA,VIAS URBANAS,MOTOCICLETA,PASSAGEIRO,MOTOCICLETA,MASCULINO,17.0,GRAVE,15 a 19,0-17,ENTREGADOR(A),BÁSICO,BRASILEIRA,01/01/2022,2022,1,1,2022/01,NaN,NaN,NaN,NaN,NaN,NaN,PUBLICO,NaN,pessoas_01-2022.csv
3,2466426,14057,15090.0,3550308,SAO PAULO,METROPOLITANA DE SÃO PAULO,VIAS URBANAS,MOTOCICLETA,CONDUTOR,MOTOCICLETA,MASCULINO,31.0,FATAL,30 a 34,30-34,NaN,NaN,NaN,01/01/2022,2022,1,1,2022/01,01/01/2022,2022.0,1.0,1.0,2022/01,ESTABELECIMENTO DE SAUDE,PUBLICO,0.0,pessoas_01-2022.csv
4,1278344,113170,98895.0,3529005,MARILIA,MARÍLIA,VIAS URBANAS,MOTOCICLETA,PASSAGEIRO,MOTOCICLETA,MASCULINO,23.0,GRAVE,20 a 24,18-24,AUXILIAR DE SERVIÇOS GERAIS,NaN,BRASILEIRA,01/01/2022,2022,1,1,2022/01,NaN,NaN,NaN,NaN,NaN,NaN,PUBLICO,NaN,pessoas_01-2022.csv


## 3. Diagnóstico ANTES

In [4]:
def diagnostico(df: pd.DataFrame) -> pd.DataFrame:
    """Tabela-resumo usada antes e depois: tipo, % de nulos e nº de valores distintos por coluna."""
    return pd.DataFrame({
        'dtype': df.dtypes.astype(str),
        'pct_nulo': (df.isna().mean() * 100).round(2),
        'n_distintos': df.nunique(),
    })

antes = diagnostico(df)
antes

,dtype,pct_nulo,n_distintos
id_sinistro,int64,0.00,132358
id_pessoa,int64,0.00,266128
id_veiculo,float64,20.09,185205
cod_ibge,int64,0.00,642
municipio,str,0.00,642
regiao_administrativa,str,0.00,16
tipo_via,str,0.00,3
tipo_veiculo_vitima,str,20.02,7
tipo_de_vitima,str,0.00,4
modo_transporte_vitima,str,19.54,8


## 4. Cruzamento: veículo nulo × tipo de vítima

In [5]:
# linhas em que o tipo de veículo da vítima está vazio
sem_veiculo = df[df['tipo_veiculo_vitima'].isna()]

# dessas, quantas são de cada tipo de vítima (pedestre, condutor, passageiro...)
print(sem_veiculo['tipo_de_vitima'].value_counts(dropna=False))

# % de linhas sem veículo no total
print(round(len(sem_veiculo) / len(df) * 100, 2))

# % de pedestres entre as linhas sem veículo
print(round((sem_veiculo['tipo_de_vitima'] == 'PEDESTRE').mean() * 100, 2))

# % de pedestres sem veículo, no total de linhas
print(round(len(sem_veiculo[sem_veiculo['tipo_de_vitima'] == 'PEDESTRE']) / len(df) * 100, 2))

tipo_de_vitima
NAO DISPONIVEL    28653
PEDESTRE          16181
PASSAGEIRO         7108
CONDUTOR           1324
Name: count, dtype: int64
20.02
30.38
6.08


## 5. Óbitos

In [6]:
# quantas linhas têm data de óbito preenchida (= quem morreu)
obitos = df['data_obito'].notna().sum()
print('óbitos:', obitos, '| % do total:', round(obitos / len(df) * 100, 2))

# as 7 colunas de óbito são nulas juntas?
# True  = em toda linha, as 7 estão TODAS vazias (não morreu) ou TODAS preenchidas (morreu)
# False = alguma linha tem óbito preenchido pela metade (dado quebrado)
cols_obito = ['data_obito', 'ano_obito', 'mes_obito', 'dia_obito',
              'ano_mes_obito', 'local_obito', 'tempo_sinistro_obito']
print((df[cols_obito].isna().all(axis=1) | df[cols_obito].notna().all(axis=1)).all())

óbitos: 5445 | % do total: 2.05
True


## 6. Valores corrompidos (acentos quebrados)

In [7]:
# colunas de texto (o pandas novo guarda texto como 'str', por isso as duas opções)
colunas_texto = df.select_dtypes(include=['object', 'str']).columns

achados = []
for coluna in colunas_texto:
    # valores diferentes da coluna, sem os vazios
    valores = pd.Series(df[coluna].dropna().unique())

    # suspeitos: letra sozinha (acento perdido, ex.: "S O PAULO") ou "Ã" (acento mal lido)
    suspeitos = valores[valores.str.contains(r'\b[B-DF-NP-TV-Z]\b|Ã', regex=True)]

    # guarda cada suspeito com o nome da coluna onde apareceu
    for v in suspeitos:
        achados.append({'coluna': coluna, 'valor': v})

# transforma a lista numa tabela
corrompidos = pd.DataFrame(achados)
print(len(corrompidos), 'valores suspeitos')

# tira os valores com acento (estão certos) e fica só com os suspeitos de verdade
reais = corrompidos[corrompidos['valor'].map(str.isascii)]
print(len(reais), 'valores realmente suspeitos')
print(reais.to_string())

# salva a lista em arquivo
reais.to_csv(RAIZ / "docs" / "perfil" / "corrompidos_2022.csv", index=False)

15 valores suspeitos
8 valores realmente suspeitos
      coluna                   valor
0  municipio   SANTA BARBARA D OESTE
1  municipio         ESTRELA D OESTE
2  municipio  SAO JOAO DO PAU D ALHO
3  municipio        PALMEIRA D OESTE
4  municipio       APARECIDA D OESTE
5  municipio     SANTA CLARA D OESTE
6  municipio      SANTA RITA D OESTE
7  municipio         GUARANI D OESTE


## 7. Roda perfil

In [8]:
# roda a função perfil(): calcula as respostas das 10 perguntas padrão e guarda em p
p = perfil(df)

## 8. Gera o arquivo

In [9]:
# grava as respostas em docs/perfil/perfil_2022.md
_ = perfil_para_markdown(p, str(SAIDA), titulo="2022")

## 9. Resumo

**O que foi feito:** perfil dos dados de 2022 (12 arquivos mensais `pessoas_*-2022.csv`), usando a função `perfil()`. Nenhum dado foi alterado; só foram feitos diagnósticos.

**Por quê:** as informações do perfil alimentam as etapas de limpeza e de tratamento de nulos e categorias (ETL-12 e ETL-16).

**Resultados:**
- 266.128 linhas × 31 colunas (mais `arquivo_origem`), com `id_pessoa` único e 132.358 sinistros, todos de 2022.
- 642 municípios e 642 códigos IBGE.
- Nulos: `grau_de_instrucao` 77,82%; `id_veiculo` 20,09%; `tipo_veiculo_vitima` 20,02%; `modo_transporte_vitima` 19,54%; `nacionalidade` 10,44%; `profissao` 9,32%; `idade` 1,43%; `local_via` 1,27%.
- As 7 colunas de óbito têm 97,95% de nulos, sempre juntas.
- Óbitos: 5.445 (2,05% do total).
- Veículo nulo × tipo de vítima: 53.266 linhas (20,02%). Delas, 53,79% são "NAO DISPONIVEL", 30,38% pedestres, 13,34% passageiros e 2,49% condutores. Passageiros e condutores sem veículo (8.432 linhas) são registros incompletos.
- Nulos disfarçados: "NAO DISPONIVEL" e "NÃO INFORMADO" aparecem como texto, não como nulo.
- Texto: os acentos estão corretos. 8 municípios que tinham apóstrofo ficaram com espaço (ex.: "SANTA BARBARA D OESTE"). `municipio` vem sem acento e `regiao_administrativa` com acento. Todos os 68 valores distintos de `nacionalidade` têm espaços sobrando no fim (238.353 linhas).

**Respostas às 10 perguntas padrão:** em `docs/perfil/perfil_2022.md`. Lista de valores suspeitos: `docs/perfil/corrompidos_2022.csv`.